# N02 — Diffraction, Gaussian propagation and numerical convergence

**Week 1 · Beginner/intermediate · 40–55 minutes · CPU / no credentials**

**Virtual apparatus:** Gaussian laser, free-space translation, beam profiler. **Observable:** propagated beam waist, integrated power and field phase.

**Model:** homogeneous scalar Fresnel and angular-spectrum propagation applied independently to two transverse Jones components. **Not:** a full-vector anisotropic Maxwell solver.

### Predict
With vacuum wavelength `λ0`, uniform refractive index `n` and waist `w0`: Rayleigh range `zR=π n w0²/λ0`; `w(z)=w0 sqrt(1+(z/zR)²)`. For an ideal Gaussian, predict `w(0.4m)`.

**Important:** PhotonicsLab removes homogeneous carrier phase `exp(ikz)`. The propagated envelope alone is *insufficient* to recombine different optical paths in an interferometer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from photonicslab import Grid, GaussianBeam, FreeSpace

wavelength = 632.8e-9
w0 = 0.25e-3
z = 0.4
grid = Grid(nx=256, ny=256, width=4e-3, height=4e-3)
source = GaussianBeam(wavelength=wavelength, waist=w0, power=1e-3).emit(grid)

def radius_x(field):
    intensity = field.intensity
    xx = field.grid.mesh[0]
    center = np.sum(xx * intensity) / intensity.sum()
    return 2 * np.sqrt(np.sum((xx-center)**2 * intensity) / intensity.sum())

zR = np.pi * w0**2 / wavelength
prediction = w0 * np.sqrt(1 + (z/zR)**2)
fresnel = FreeSpace(z, method="fresnel").apply(source)
angular = FreeSpace(z, method="angular-spectrum").apply(source)
print(f"zR={zR:.6f} m, predicted waist={prediction*1e6:.3f} µm")
for name, field in [("Fresnel",fresnel),("Angular spectrum",angular)]:
    print(f"{name}: waist={radius_x(field)*1e6:.3f} µm, power={field.power*1e3:.6f} mW")
    assert np.isclose(radius_x(field), prediction, rtol=3e-3)
    assert np.isclose(field.power, source.power, rtol=1e-11)


### From the Fresnel equation to a numerical operator

For the adopted `exp(i kz-iωt)` convention, the paraxial transfer function for an envelope is

`H(fx,fy;z)=exp[-i π (λ0/n) z (fx²+fy²)]`.

Below we implement this **once from first principles** to verify the library's Fresnel operator on the *same sampled periodic grid*. This agreement establishes an implementation cross-check, not an independent experimental validation.


In [ ]:
fx = np.fft.fftfreq(grid.nx, d=grid.dx)
fy = np.fft.fftfreq(grid.ny, d=grid.dy)
FX, FY = np.meshgrid(fx, fy, indexing="xy")
H = np.exp(-1j * np.pi * wavelength * z * (FX**2 + FY**2))
manual_ex = np.fft.ifft2(np.fft.fft2(source.amplitude[0]) * H)
np.testing.assert_allclose(fresnel.amplitude[0], manual_ex, rtol=1e-11, atol=1e-11)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(grid.x*1e3, source.intensity[grid.ny//2], label="waist plane")
ax.plot(grid.x*1e3, fresnel.intensity[grid.ny//2], label=f"after {z:g} m")
ax.set(xlabel="x (mm)", ylabel="Irradiance (W/m²)", title="Diffraction expands the beam")
ax.legend()
plt.show()


### Convergence, bad boundaries and validity

An FFT implicitly imposes a periodic computational window. The integral of power may still be numerically conserved while an expanding field wraps around an undersized domain. Compare a larger grid and deliberately poor transverse window.

The Fresnel and angular-spectrum transfer functions approach each other in a paraxial regime, but they are not universally equivalent: examine their assumptions before using a high-angle beam or near-field subwavelength structure.


In [ ]:
fine_grid = Grid(nx=384, ny=384, width=4e-3, height=4e-3)
fine_source = GaussianBeam(wavelength=wavelength, waist=w0, power=1e-3).emit(fine_grid)
fine = FreeSpace(z, method="fresnel").apply(fine_source)
poor_grid = Grid(nx=128, ny=128, width=0.6e-3, height=0.6e-3)
poor_source = GaussianBeam(wavelength=wavelength, waist=w0, power=1e-3).emit(poor_grid)
poor = FreeSpace(z, method="fresnel").apply(poor_source)
print(f"256 -> 384 grid relative waist difference: {abs(radius_x(fine)-radius_x(fresnel))/prediction:.3g}")
print(f"good edge fraction: {fresnel.edge_power_fraction():.3g}; bad: {poor.edge_power_fraction():.3g}")
assert abs(radius_x(fine)-radius_x(fresnel))/prediction < 2e-3
assert poor.edge_power_fraction() > fresnel.edge_power_fraction()


### Lab translation, failure exercise and assessment

**Bench:** He–Ne laser or appropriately approved equivalent, adjustable propagation distance, calibrated beam profiler, neutral-density filters and beam dump. For each position record spot width and uncertainty; compare `w(z)` to the Gaussian law. Account for beam ellipticity, waist position, detector saturation and apertures.

**Failure task:** make the FFT window too small while keeping reported power fixed. Explain how power conservation can coexist with a physically wrong irradiance pattern. Estimate the needed padding and `dx` for your selected distance.

**Evidence to save:** `λ0`, `n`, `w0`, distance, both transfer-function choices, grid/window refinement, waist-vs-distance graph, uncertainty and code/environment revision.

**Limitations:** monochromatic coherent scalar propagation, paraxial Fresnel or scalar Helmholtz angular spectrum, periodic boundaries and no anisotropy. Angular-spectrum implementation includes forward evanescent decay, but not a complete vector Poynting-flux model.

**Read:** Goodman, *Introduction to Fourier Optics*; Matsushima & Shimobaba, *Optics Express* (2009), DOI: https://doi.org/10.1364/OE.17.019662. **Next:** N03 Jones optics and Stokes parameters.
